# SAND (Spatial Adjacency Network Detection)
SAND identifies local structures ('clusters') in spatial data based on user defined values


In [1]:
from requirements import *
from scipy.cluster.hierarchy import DisjointSet
from sklearn.neighbors import NearestNeighbors
import scipy

In [2]:
"""
SAND: identification of local structures in spatial data
Inputs  1) adata_xenv1: anndata object with cell type annotations under .obs['ct'], and x,y spatial coordinates under .obsm['spatial']
        2) k: number of nearest neighbors for KNN graph
        3) r: distance cutoff between two cells to filter KNN graph when k is defined or for the construction of a radius-based graph
        4) valuelist: list of cell types of interest (optional)
        
Returns 1) Annotates adata_xenv1 object input with cluster assignments in .obs['cluster']
"""
def runSAND(adata_xenv1, k = None, r = None, valuelist = []):
    if k is None and r is None:
        raise ValueError("Define at least one: k, r")
 
    filtered_neighs = makeneighborgraph(adata_xenv1, k, r, valuelist)
    joinsets(adata_xenv1, filtered_neighs)
    if len(valuelist):
        adata_xenv1.obs.loc[~adata_xenv1.obs['ct'].isin(valuelist), 'cluster'] = 'NA'
    
def makeneighborgraph(adata_xenv1, k, r, valuelist):
    keepidx = [i for i in range(adata_xenv1.shape[0])]
    if len(valuelist):
        keepidx = [i for i,c in enumerate(adata_xenv1.obs['ct']) if c in valuelist]              
    map_idxtoct = dict(map(lambda i,j : (i,j) , keepidx,[adata_xenv1.obs['ct'][x] for x in keepidx])) 
    
    if k != None: # KNN graph
        knn = NearestNeighbors(n_neighbors=k)
        knn.fit(adata_xenv1.obsm['spatial'])
        neighbors = knn.kneighbors()

        r_filter = r
        if r is None:
            r_filter = float('inf')
        
        filtered_neighs = dict() 
        for idx in keepidx:
            filtered_neighs[idx] = set([
                j for i,j in enumerate(neighbors[1][idx])
                if (j in map_idxtoct) and (neighbors[0][idx][i] < r_filter) # filter neighbors by cell type and radius
            ]) 
    
        return filtered_neighs
    
    else: # radius graph
        tree = scipy.spatial.KDTree(adata_xenv1.obsm['spatial']) 
        neighbors = tree.query_ball_tree(tree, r) 
    
        filtered_neighs = dict()
        for idx in keepidx:
            filtered_neighs[idx] = set([
                j for j in neighbors[idx]
                if j in map_idxtoct and j != idx
            ]) 
            
        return filtered_neighs

def joinsets(adata_xenv1, filtered_neighs): # call union find
    edges = set()
    nodes = set()
    for k in filtered_neighs:
        nodes.add(k)
        for v in filtered_neighs[k]:
            edges.add((k,v))
            
    disjoint_set = DisjointSet(nodes)
    for a,b in edges:
        disjoint_set.merge(a, b)

    clusters = dict() # key is cell; value is cluster assignment (root cell)
    for s in disjoint_set.subsets():
        for c in s:
            clusters[c] = disjoint_set.__getitem__(c)

    adata_xenv1.obs['cluster'] = range(adata_xenv1.shape[0])
    adata_xenv1.obs = adata_xenv1.obs.replace({"cluster": clusters})
    

In [3]:
# Load data
adata_xenv1 = sc.read_h5ad(XEN_V1)

In [4]:
adata_xenv1.obsm['spatial']

array([[ 8052.32246051,  5706.30418283],
       [ 8289.39414416, 11536.8386676 ],
       [ 8572.75282306,  4194.90862459],
       ...,
       [ 6443.40314207,  4621.58353301],
       [ 7584.16259177,  8600.23987635],
       [ 9566.17937151,  7958.67392972]])

In [8]:
# Run SAND for each sample to find immune aggregates
adata_xenv1.obs['ct'] = adata_xenv1.obs['ct_general']

sand_di = dict()
for samp in adata_xenv1.obs['sample_name'].unique():
    print(samp)
    ad = adata_xenv1[adata_xenv1.obs['sample_name'] == samp].copy()
    samp = str(samp)
    runSAND(ad, 10, 15, ['T Cell', 'B Cell', 'pDC', 'NK'])
    for i in range(ad.shape[0]):
        clustid = str(ad.obs['cluster'][i])
        if clustid != 'NA':
            clustid = clustid + '_' + samp
        sand_di[ad.obs.index[i]] = clustid

adata_xenv1.obs['cluster'] = [sand_di[x] for x in adata_xenv1.obs.index]

region1_0011549
region1_0011552
region2_0011552
region3_0011552
region1_0027612
region1_0027609
region2_0015400
region1_0015399
region1_0027860
region2_0027861
region1_0043106
region1_0043114
region1_0036509
region1_0036579
region1_0042581
region1_0033583
region2_0033583
region2_0042591
region1_0042592
region1_0064838
region2_0064838
region1_0065604
region4_0065604
region1_0065607
region2_0065607
region1_0069145
region2_0069145
region1_0069146
region2_0069146
region1_0069319
region2_0069319
region1_0069320
region2_0069320
region1_0069330
region1_0069331
region1_0069670
region1_0069737
region2_0069737
region1_0069187
region1_0069478
region2_0069478
region1_0069480
region1_0069594
region2_0069594
region1_0069595
region2_0069595
region1_0069641
region2_0069641
region1_0069808
region2_0015858
region1_0015858
region1_0015829
region2_0015829
region2_0069665
region1_0069665
region3_0078989
region2_0078989
region3_0078981
region1_0078989
region1_0078981
region2_0078981
region1_0079272
region2_

In [9]:
# Size of immune aggregates
adata_xenv1.obs['cluster'].value_counts()

NA                        5317295
277612_region1_0043106       1026
971_region1_0069594           968
16737_region1_0069320         817
275691_region1_0043106        764
                           ...   
42591_region1_0065607           1
42711_region1_0065607           1
42728_region1_0065607           1
42770_region1_0065607           1
43416_region1_0079269           1
Name: cluster, Length: 259066, dtype: int64

In [12]:
# Cell types within largest immune aggregate
adata_xenv1.obs['ct'][adata_xenv1.obs['cluster'] == '277612_region1_0043106'].value_counts()

T Cell                 444
B Cell                 411
pDC                    160
NK                      11
Megakaryocyte            0
VSMC                     0
Plasma Cell              0
Osteoblast               0
Mesenchymal/Stromal      0
Adipocyte                0
Macrophage               0
Late/Mature Myeloid      0
HSPC                     0
Erythroid                0
Endothelial              0
Early Myeloid            0
Mast                     0
Name: ct, dtype: int64